# Model Competition and Variable Selection
## ML Modeling Challenge - Wizeline

Entrenamiento y selección de modelos candidatos:
- **Lasso** (baseline lineal)
- **DecisionTree**
- **RandomForest**
- **ExtraTrees**
- **GradientBoosting**
- **XGBoost**
- **LightGBM**
- **CatBoost**

El mejor modelo se selecciona por el menor **SMAPE de Cross-Validation**.

Se hace selección de variables definitiva vía features importances.


# 0. Imports y configuración.

In [1]:
import logging

import plotly.express as px
import yaml
import polars as pl

from src.train_functions import (
    evaluate_all_models,
    get_feature_importances,
    get_metric_cv_comparison,
    get_model_pipelines,
    plot_feature_importances,
    run_cross_validation,
)

logging.basicConfig(level=logging.INFO, format="%(message)s")

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

SEED: int = config["model"]["seed"]
CV_FOLDS: int = config["model"]["cv_folds"]
LIST_FEATURE_SELECTED: list[str] = config["features"]["selected_by_mi"]

print(f"Seed global: {SEED}")
print(f"CV folds: {CV_FOLDS}")
print(f"Features pre-seleccionadas: {LIST_FEATURE_SELECTED}")


Seed global: 5000
CV folds: 5
Features pre-seleccionadas: ['feature_2', 'feature_13', 'feature_16', 'feature_9', 'feature_3', 'feature_18', 'feature_11', 'feature_5', 'feature_1', 'feature_12', 'feature_15', 'feature_4']


# 1. Carga de datos y selección de features.

In [2]:
df_train = pl.read_csv("data/training_data.csv")

In [3]:
X = df_train.select(LIST_FEATURE_SELECTED).to_numpy()
y = df_train["target"].to_numpy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (800, 12)
y shape: (800,)


# 2. Construcción de pipelines.

In [4]:
PIPELINES = get_model_pipelines(seed=SEED)

print(f"Modelos a evaluar: {list(PIPELINES.keys())}")

Modelos a evaluar: ['Lasso', 'DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost']


__NOTAS:__
1. Cada pipeline aplica **MinMaxScaler** seguido del estimador, esto para dejar todas las features en la misma escala.
2. Todos los modelos estocásticos usan `random_state=SEED` (5000) para reproducibilidad.
3. Por ahora, todos los modelos serán evaluados usando los valores por default de sus hiperparámetros.
3. Aquí algunas ideas para mi elección de modelos:
    * `Lasso`: Modelo lineal base. Se usa Lasso para eliminar variables innecesarias mediante regularización L1.
    * `DecisionTree`: Primera opción para capturar complejidades no lineales. Si queda como el mejor, se estará obedeciendo al principio de Navaja de Ockham. Recordar que se puede observar alta varianza y sobreajuste.
    * `RandomForest`: Evolución natural del DecisionTree usando bagging (reduce la varianza promediando árboles decorrelacionados).
    * `ExtraTrees`: Similar a RandomForest pero con splits completamente aleatorios (se reduce aún más la varianza y es más rápido). 
    * `GradientBoosting`: Referencia de *boosting* puro de sklearn. Construye árboles secuencialmente corrigiendo los residuos del anterior. 
    * `XGBoost`: Mi "Caballo de troya" por experiencia. Versión optimizada del boosting base (regularización L1/L2 nativa y eficiencia computacional). 
    * `LightGBM`, `CatBoost`: Variantes optimizadas de boosting competidores directos de `XGBoost`.

# 3. Cross-Validation (k=4).

Métricas calculadas por fold y promediadas:
- **RMSE** — Root Mean Squared Error
- **MAE** — Mean Absolute Error
- **MAPE** — Mean Absolute Percentage Error
- **SMAPE** — Symmetric MAPE *(métrica de selección)*
- **FUGACITY_SMAPE** - Porcentaje de predicciones cuyo error SMAPE individual supera el 15% (tasa de predicciones con error grande)
- **R²** — Coeficiente de determinación


In [5]:
df_cv = evaluate_all_models(PIPELINES, X, y, n_splits=CV_FOLDS, seed=SEED)

Evaluating Lasso...
Lasso — RMSE=5.1087 | MAE=4.2083 | SMAPE=30.5543% | Fug_SMAPE=69.7500% | R²=-0.0278
Evaluating DecisionTree...
DecisionTree — RMSE=3.5639 | MAE=2.8271 | SMAPE=22.3894% | Fug_SMAPE=53.0000% | R²=0.4973
Evaluating RandomForest...
RandomForest — RMSE=2.4599 | MAE=1.9335 | SMAPE=15.0847% | Fug_SMAPE=37.8750% | R²=0.7620
Evaluating ExtraTrees...
ExtraTrees — RMSE=2.3360 | MAE=1.8758 | SMAPE=14.7759% | Fug_SMAPE=36.1250% | R²=0.7850
Evaluating GradientBoosting...
GradientBoosting — RMSE=2.1124 | MAE=1.6924 | SMAPE=13.4014% | Fug_SMAPE=31.1250% | R²=0.8240
Evaluating XGBoost...
XGBoost — RMSE=2.3699 | MAE=1.8921 | SMAPE=15.0803% | Fug_SMAPE=35.8750% | R²=0.7787
Evaluating LightGBM...
/home/miguel_uicab/DOCUMENTS/ML_Modeling_Challenge/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/miguel_uicab/DOCUMENTS/ML_Modeling_Challenge/.venv/li

# 4. Comparación de modelos.

In [6]:
fig = get_metric_cv_comparison(df_cv, sort_by="smape_cv" , width=1300)
fig.show()

__NOTAS:__
1. xd

# 5. Selección del mejor modelo

El **modelo ganador** es el que tiene el menor **SMAPE de CV**.

In [7]:
best_row = df_cv.sort("smape_cv").row(0, named=True)
best_model_name: str = best_row["model"]
best_smape: float = best_row["smape_cv"]

print(f"Modelo ganador : {best_model_name}")
print(f"SMAPE CV       : {best_smape:.4f} %")
print(f"FUGACITY CV    : {best_row['fugacity_smape_cv']:.4f} %")
print(f"R² CV          : {best_row['r2_cv']:.4f}")

Modelo ganador : CatBoost
SMAPE CV       : 11.8019 %
FUGACITY CV    : 23.8750 %
R² CV          : 0.8684


# 6. Feature importances del modelo ganador

Se entrena el modelo ganador con **todos los datos de entrenamiento** y se grafican las importancias.

In [8]:
best_pipeline = PIPELINES[best_model_name]
best_pipeline.fit(X, y)

df_importances = get_feature_importances(best_pipeline, LIST_FEATURE_SELECTED)

In [9]:
fig = plot_feature_importances(df_importances, model_name=best_model_name)
fig.show()

# 7. Segunda competencia con selección de features.

Se filtran del modelo ganador (sección 6) las features con **importancia > 5** y se repite la evaluación completa: Cross-Validation → comparación → mejor modelo.

In [10]:
IMPORTANCE_THRESHOLD = 5.0

LIST_FEATURE_SELECTED_V2 = (
    df_importances
    .filter(pl.col("importance") > IMPORTANCE_THRESHOLD)
    ["feature"]
    .to_list()
)

print(f"Features con importancia > {IMPORTANCE_THRESHOLD}: {len(LIST_FEATURE_SELECTED_V2)}")
print(LIST_FEATURE_SELECTED_V2)

Features con importancia > 5.0: 5
['feature_2', 'feature_13', 'feature_9', 'feature_11', 'feature_18']


In [11]:
X_v2 = df_train.select(LIST_FEATURE_SELECTED_V2).to_numpy()

print(f"X_v2 shape: {X_v2.shape}")

X_v2 shape: (800, 5)


### 7.1 Cross-Validation (k=4).


In [12]:
df_cv_v2 = evaluate_all_models(
    get_model_pipelines(seed=SEED), X_v2, y, n_splits=CV_FOLDS, seed=SEED, label="v2"
)

Evaluating Lasso...
Lasso — RMSE=5.1087 | MAE=4.2083 | SMAPE=30.5543% | Fug_SMAPE=69.7500% | R²=-0.0278
Evaluating DecisionTree...
DecisionTree — RMSE=3.2559 | MAE=2.5568 | SMAPE=20.1624% | Fug_SMAPE=48.7500% | R²=0.5826
Evaluating RandomForest...
RandomForest — RMSE=2.2680 | MAE=1.7849 | SMAPE=14.0323% | Fug_SMAPE=33.8750% | R²=0.7975
Evaluating ExtraTrees...
ExtraTrees — RMSE=2.1881 | MAE=1.7533 | SMAPE=13.9376% | Fug_SMAPE=32.2500% | R²=0.8114
Evaluating GradientBoosting...
GradientBoosting — RMSE=2.0173 | MAE=1.6218 | SMAPE=12.9551% | Fug_SMAPE=29.8750% | R²=0.8396
Evaluating XGBoost...
XGBoost — RMSE=2.1807 | MAE=1.7313 | SMAPE=13.8587% | Fug_SMAPE=34.0000% | R²=0.8113
Evaluating LightGBM...
/home/miguel_uicab/DOCUMENTS/ML_Modeling_Challenge/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/miguel_uicab/DOCUMENTS/ML_Modeling_Challenge/.venv/li

### 7.2 Comparación de modelos.

In [13]:
fig = get_metric_cv_comparison(df_cv_v2, sort_by="smape_cv" , width=1300)
fig.show()

### 7.3 Mejor modelo (v2)

In [14]:
best_row_v2 = df_cv_v2.sort("smape_cv").row(0, named=True)
best_model_name_v2: str = best_row_v2["model"]
best_smape_v2: float = best_row_v2["smape_cv"]
best_fugacity_smape_v2: float = best_row_v2["fugacity_smape_cv"]

print(f"Modelo ganador (v2) : {best_model_name_v2}")
print(f"SMAPE CV            : {best_smape_v2:.4f}%")
print(f"FUGACITY CV         : {best_fugacity_smape_v2:.4f} %")
print(f"R² CV               : {best_row_v2['r2_cv']:.4f}")

print("\n--- Comparación v1 vs v2 ---")
print(f"SMAPE v1 ({best_model_name:<10}): {best_smape:.4f}%")
print(f"SMAPE v2 ({best_model_name_v2:<10}): {best_smape_v2:.4f}%")
pct_smape = (best_smape_v2 - best_smape) / best_smape * 100
print(f"Mejora              : {pct_smape:+.2f}% ({'mejor' if pct_smape < 0 else 'peor'} con features filtradas)")

print(f"\nFUGACITY v1 ({best_model_name:<10}): {best_row['fugacity_smape_cv']:.4f}%")
print(f"FUGACITY v2 ({best_model_name_v2:<10}): {best_fugacity_smape_v2:.4f}%")
pct_fugacity = (best_fugacity_smape_v2 - best_row['fugacity_smape_cv']) / best_row['fugacity_smape_cv'] * 100
print(f"Mejora              : {pct_fugacity:+.2f}% ({'mejor' if pct_fugacity < 0 else 'peor'} con features filtradas)")

print(f"\nR² v1 ({best_model_name:<10}): {best_row['r2_cv']:.4f}")
print(f"R² v2 ({best_model_name_v2:<10}): {best_row_v2['r2_cv']:.4f}")
pct_r2 = (best_row_v2['r2_cv'] - best_row['r2_cv']) / abs(best_row['r2_cv']) * 100
print(f"Mejora              : {pct_r2:+.2f}% ({'mejor' if pct_r2 > 0 else 'peor'} con features filtradas)")


Modelo ganador (v2) : CatBoost
SMAPE CV            : 11.3860%
FUGACITY CV         : 23.5000 %
R² CV               : 0.8802

--- Comparación v1 vs v2 ---
SMAPE v1 (CatBoost  ): 11.8019%
SMAPE v2 (CatBoost  ): 11.3860%
Mejora              : -3.52% (mejor con features filtradas)

FUGACITY v1 (CatBoost  ): 23.8750%
FUGACITY v2 (CatBoost  ): 23.5000%
Mejora              : -1.57% (mejor con features filtradas)

R² v1 (CatBoost  ): 0.8684
R² v2 (CatBoost  ): 0.8802
Mejora              : +1.36% (mejor con features filtradas)


### 7.4 Feature importances (v2)

In [15]:
best_pipeline_v2 = get_model_pipelines(seed=SEED)[best_model_name_v2]
best_pipeline_v2.fit(X_v2, y)

df_importances_v2 = get_feature_importances(best_pipeline_v2, LIST_FEATURE_SELECTED_V2)

In [16]:
fig = plot_feature_importances(df_importances_v2, model_name=best_model_name_v2)
fig.show()

# 8. Guardado de archivos

Se actualiza `config.yaml` con los resultados finales del proceso de selección:
- `features.selected_v2` — features seleccionadas por importancia (v2), ordenadas de mayor a menor
- `model.name` — nombre del modelo ganador en v2
- `model.cv_metrics_v2` — métricas de CV del modelo ganador: `smape_cv`, `r2_cv`, `fugacity_smape_cv`

In [17]:
CONFIG_PATH = "config.yaml"

# Features ordenadas de mayor a menor importancia (v2)
features_v2_by_importance: list[str] = (
    df_importances_v2
    .sort("importance", descending=True)
    ["feature"]
    .to_list()
)

with open(CONFIG_PATH, "r") as f:
    config_to_save = yaml.safe_load(f)

config_to_save["features"]["selected_v2"] = features_v2_by_importance
config_to_save["model"]["name"] = best_model_name_v2
config_to_save["model"]["cv_metrics_v2"] = {
    "smape_cv": round(best_smape_v2, 6),
    "r2_cv": round(best_row_v2["r2_cv"], 6),
    "fugacity_smape_cv": round(best_fugacity_smape_v2, 6),
}

with open(CONFIG_PATH, "w") as f:
    yaml.dump(config_to_save, f, allow_unicode=True, sort_keys=False)

print(f"config.yaml actualizado:")
print(f"  model.name               : {best_model_name_v2}")
print(f"  features.selected_v2     : {features_v2_by_importance}")
print(f"  model.cv_metrics_v2.smape_cv         : {best_smape_v2:.6f}")
print(f"  model.cv_metrics_v2.r2_cv            : {best_row_v2['r2_cv']:.6f}")
print(f"  model.cv_metrics_v2.fugacity_smape_cv: {best_fugacity_smape_v2:.6f}")

config.yaml actualizado:
  model.name               : CatBoost
  features.selected_v2     : ['feature_2', 'feature_13', 'feature_9', 'feature_18', 'feature_11']
  model.cv_metrics_v2.smape_cv         : 11.385998
  model.cv_metrics_v2.r2_cv            : 0.880168
  model.cv_metrics_v2.fugacity_smape_cv: 23.500000
